# Gemma 4 Developer Agent — Competition Optimized

This notebook builds a valid `submission.zip` for the Gemma 4 Developer Agent Competition. The scorer starts its own model server, executes the declared agent, captures a patch, and grades the patch against hidden tests.

## Baseline diagnosis
The attached notebook loaded 129 tasks but only smoke-tested two. Both saved runs returned `NO_PATCH` (0/2), with one-minute task limits; one exhausted its 10-call budget and the other timed out before making a patch. That is a diagnostic failure, not a reliable estimate of the full leaderboard score.

## Optimizations
- Quality-first default: one official Gemma 4 31B quantized model across all agents; smaller allowed variants selectable by environment variable.
- Remove the repository's tiny starter LoRA placeholders unless replaced with a genuinely trained adapter for the selected base model.
- Evidence-first agent prompt: use task clues, focused search, source inspection, targeted tests, a minimal patch, and explicit `submit_patch()`.
- Per-task budget raised to 4.5 minutes, 28 paid tool calls, 96 turns; command timeout 180 seconds. Across 129 tasks, the wall-clock allowance totals about 9.7 hours, leaving headroom beneath the 12-hour generation window.
- Validated ZIP: root `agent.yaml`, required prompt/include paths, allowed extensions, no symlinks, CRC check, and less than 3 GiB unpacked size.

**No notebook can guarantee a 1.0 score:** each task must pass hidden tests. This version has not been run against Kaggle's hidden evaluation.

## 1. Select the model and prepare the submission directory

Set `GEMMA_MODEL_NAME` to one of the four allowed quantized variants. The default is 31B for quality; choose 12B/E4B/E2B only after measuring the quality-throughput trade-off. Leave `KEEP_TRAINED_ADAPTERS=0` for the starter adapters.

In [ ]:

import json
import os
import re
import shutil
from pathlib import Path

DATA_DIR = Path("/kaggle/input/competitions/gemma-4-developer-agent")
WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)

ALLOWED_MODELS = {
    "gemma-4-31b-it-qat-w4a16-ct",
    "gemma-4-12b-it-qat-w4a16-ct",
    "gemma-4-e4b-it-qat-w4a16-ct",
    "gemma-4-e2b-it-qat-w4a16-ct",
}
MODEL_NAME = os.environ.get("GEMMA_MODEL_NAME", "gemma-4-31b-it-qat-w4a16-ct").strip()
if MODEL_NAME not in ALLOWED_MODELS:
    raise ValueError(f"Unsupported model {MODEL_NAME!r}; choose from {sorted(ALLOWED_MODELS)}")

KEEP_TRAINED_ADAPTERS = os.environ.get("KEEP_TRAINED_ADAPTERS", "0").lower() in {"1", "true", "yes"}
sample_src = DATA_DIR / "sample_submission"
if not sample_src.is_dir():
    raise FileNotFoundError(f"Missing competition sample submission: {sample_src}")

AGENT_DIR = WORKING_DIR / "gemma4_agent"
if AGENT_DIR.exists():
    shutil.rmtree(AGENT_DIR)
shutil.copytree(sample_src, AGENT_DIR)

# Every agent must declare exactly the same permitted base model.
yaml_paths = [AGENT_DIR / "agent.yaml", *sorted((AGENT_DIR / "sub_agents").glob("*.yaml"))]
for path in yaml_paths:
    if not path.is_file():
        continue
    text = path.read_text(encoding="utf-8")
    text = re.sub(r"(?m)^([ \t]*model:[ \t]*).+$", rf"\g<1>{MODEL_NAME}", text)
    if not KEEP_TRAINED_ADAPTERS:
        text = re.sub(r"(?m)^[ \t]*adapter:[^\n]*\n?", "", text)
    path.write_text(text, encoding="utf-8")

if not KEEP_TRAINED_ADAPTERS:
    shutil.rmtree(AGENT_DIR / "adapters", ignore_errors=True)
else:
    configs = list((AGENT_DIR / "adapters").glob("*/adapter_config.json"))
    if not configs:
        raise FileNotFoundError("KEEP_TRAINED_ADAPTERS=1 but no adapter configs exist.")
    for path in configs:
        base = str(json.loads(path.read_text(encoding="utf-8")).get("base_model_name_or_path", ""))
        if MODEL_NAME not in base:
            raise ValueError(f"Adapter {path} targets {base!r}, not {MODEL_NAME!r}.")

assert (AGENT_DIR / "agent.yaml").is_file()
assert yaml_paths[0].is_file()
print(f"Prepared {AGENT_DIR} with {MODEL_NAME}; retained adapters={KEEP_TRAINED_ADAPTERS}")


## 2. Use a disciplined repair prompt and competition-sized budget

The root agent and read-only analyzer use the same base model. The agent is directed to avoid broad repository scans, keep tests/configuration untouched, perform a narrow verification, inspect the final diff, and explicitly submit a non-empty patch.

In [ ]:

SYSTEM_PROMPT = """
You are an expert software engineer repairing a repository snapshot in /workspace.
Goal: produce the smallest correct source patch that satisfies the full task and passes hidden validation tests.

RULES
- Treat the complete problem statement and hints as the specification. Preserve public APIs and repository conventions.
- Work only in /workspace. The sandbox is offline: never install packages or access the network.
- Modify source/implementation files only. Never create, edit, or delete tests or runner configuration; these changes are stripped by the evaluator.
- Read exact source before editing. Do not invent filenames, behavior, or test results.

WORKFLOW
1. Extract required behavior, named symbols/files, error messages, and edge cases.
2. If a path/symbol is named, read it directly. Otherwise use one narrow `rg -n` search in likely folders. Never use `ls -R` or broad repeated `grep -R`.
3. Use graph tools selectively, only after identifying real symbols. Source files are authoritative. Delegate to the read-only analyzer only for genuinely cross-file ambiguity.
4. Make one focused patch using existing helpers. Avoid unrelated refactors and new dependencies.
5. Run the narrowest relevant existing test or targeted assertion. Never run bare pytest or a whole-repository suite. Do not leave scratch files in /workspace.
6. Run `git diff --check`, inspect `git status --short` and `git diff --stat`, and confirm the patch is non-empty and source-only.
7. Explicitly call `submit_patch()` as the final tool action. It is free and required. Do not finish with only a verbal report.

BUDGET
- Target 10–18 paid tool calls; reserve time for verification and submit_patch.
- `get_status()` and `submit_patch()` are free. Check status before long tests or when about halfway through.
- Prefer high-information reads over exploratory scans. If time is low, implement the smallest defensible fix, verify once, inspect the diff, and submit.

FINAL RESPONSE
Report files changed, behavior fixed, checks actually run, and remaining uncertainty. Never claim a test passed without its output.
""".strip()

ANALYZER_PROMPT = """
You are a read-only code investigation specialist. Inspect only files and symbols relevant to the task. Use graph tools only with identified symbols; source code is authoritative.
Return: (1) likely source files and exact symbols, (2) supported caller/callee flow, (3) facts vs root-cause hypothesis, (4) nearest existing pattern/test location, (5) minimal patch direction and compatibility risks.
Do not edit files, run broad searches, or run tests. State what cannot be determined from the snapshot.
""".strip()

(AGENT_DIR / "prompts").mkdir(parents=True, exist_ok=True)
(AGENT_DIR / "prompts" / "system.md").write_text(SYSTEM_PROMPT + "\n", encoding="utf-8")
(AGENT_DIR / "prompts" / "analyzer.md").write_text(ANALYZER_PROMPT + "\n", encoding="utf-8")

(AGENT_DIR / "configs" / "sampling.yaml").write_text(
    "temperature: 0.1\ntop_p: 0.9\ntop_k: 40\nmax_output_tokens: 12288\n"
    "thinking_config:\n  thinking_budget: 4096\n  include_thoughts: true\n",
    encoding="utf-8",
)
(AGENT_DIR / "eval_config.yaml").write_text(
    "# Per-task generation budget. 4.5 min x 129 tasks ~= 9.7 hours.\n"
    "evaluation:\n  timeout_seconds: 180\n  max_tool_calls: 28\n"
    "  max_time_minutes: 4.5\n  max_turns: 96\n",
    encoding="utf-8",
)

for path in yaml_paths:
    if path.is_file():
        declared = re.findall(r"(?m)^\s*model:\s*(\S+)", path.read_text(encoding="utf-8"))
        if declared and set(declared) != {MODEL_NAME}:
            raise AssertionError(f"Inconsistent model declarations in {path}: {declared}")
assert (AGENT_DIR / "sub_agents" / "code_analyzer.yaml").is_file()
print("Agent prompts and budgets prepared.")


## 3. Validate and build `submission.zip`

The competition expects the declarative submission with `agent.yaml` at the ZIP root. This cell checks the archive structure and the harness file constraints before writing the artifact.

In [ ]:

import zipfile

ALLOWED_EXTENSIONS = {".yaml", ".yml", ".md", ".txt", ".py", ".json", ".safetensors"}
MAX_UNPACKED_BYTES = 3 * 1024 * 1024 * 1024

assert AGENT_DIR.is_dir() and (AGENT_DIR / "agent.yaml").is_file()
required = [
    "agent.yaml",
    "eval_config.yaml",
    "configs/sampling.yaml",
    "prompts/system.md",
    "prompts/analyzer.md",
    "sub_agents/code_analyzer.yaml",
]
missing = [name for name in required if not (AGENT_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"Missing submission files: {missing}")

files = []
unpacked = 0
for path in AGENT_DIR.rglob("*"):
    if path.is_symlink():
        raise ValueError(f"Symlink not allowed: {path.relative_to(AGENT_DIR)}")
    if path.is_file():
        if path.suffix.lower() not in ALLOWED_EXTENSIONS:
            raise ValueError(f"Disallowed extension: {path.relative_to(AGENT_DIR)}")
        files.append(path)
        unpacked += path.stat().st_size
if unpacked > MAX_UNPACKED_BYTES:
    raise ValueError(f"Unpacked submission exceeds 3 GiB: {unpacked}")

zip_path = WORKING_DIR / "submission.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for path in files:
        zf.write(path, path.relative_to(AGENT_DIR).as_posix())

with zipfile.ZipFile(zip_path) as zf:
    assert zf.testzip() is None, "ZIP CRC test failed"
    names = set(zf.namelist())
    assert "agent.yaml" in names, "agent.yaml must be at ZIP root"
    assert all(not n.startswith("gemma4_agent/") for n in names)
    assert sum(info.file_size for info in zf.infolist()) <= MAX_UNPACKED_BYTES

print(f"Created {zip_path} | model={MODEL_NAME} | files={len(files)}")
print(f"Unpacked={unpacked / (1024 * 1024):.2f} MiB | ZIP={zip_path.stat().st_size / (1024 * 1024):.2f} MiB")


## Submission notes

- Run the notebook on Kaggle with the competition data and one of the permitted Gemma 4 quantized model inputs attached.
- Download `/kaggle/working/submission.zip` and submit it through the competition workflow.
- For a quality/throughput comparison, clone this notebook and test 31B vs 12B/E4B using identical task subsets and budgets. Track resolved count, non-empty patches, tool calls, elapsed time, and failure reason. Do not infer a score from two tasks.
- Keep adapters disabled unless you have actually trained and validated a LoRA against the chosen base model.
